# 02 Clean: build the municipality-year panel
Reads the `raw_*` tables from notebook 01 and writes `clean_*` tables. Decisions applied (see the README's Data decisions):
1. **Province codes:** KZ → KZN and LIM → LP (one code per province).
2. **Blank irregular-expenditure amounts** → 0 with a `was_blank` flag (decision 7).
3. **Missing audited statements** → kept as flags `has_audited_*`, not dropped or imputed (decision 5).
4. **No rows deleted:** all years kept; `mscoa_era` marks 2018+ for analyses that need consistent financials (decision 8).

The last cell runs reconciliation checks: every check must print PASS.

In [ ]:
from pyspark.sql import functions as F, Window  # Spark's toolbox of column functions (F) and window functions

TABLES = ["municipalities", "audit_opinions", "uifwexp", "incexp_v2", "financial_position_v2", "aged_debtor_v2"]  # the six raw tables loaded by notebook 01
raw = {t: spark.table(f"raw_{t}") for t in TABLES}  # open each raw table and keep it in a dictionary, e.g. raw["uifwexp"]
for t, df in raw.items():  # go through each raw table
    print(f"raw_{t}: {df.count():,} rows")  # print its row count, to check against the source

## 1. Municipalities: one code per province

In [ ]:
PROVINCE_FIX = {"KZ": "KZN", "LIM": "LP"}  # the source uses two codes for two provinces; map the odd ones to the standard code
fix = F.create_map(*[F.lit(x) for kv in PROVINCE_FIX.items() for x in kv])  # turn that dictionary into a lookup Spark can use inside a table

clean_muni = (raw["municipalities"]  # start from the raw municipalities table
    .select(F.col("municipality_demarcation_code").alias("demarcation_code"),  # the municipality's code, with a shorter name
            F.col("municipality_name").alias("name"),  # its name
            F.coalesce(fix[F.col("municipality_province_code")], F.col("municipality_province_code")).alias("province_code"),  # the province code: the fixed code if it's in the lookup, otherwise the original
            F.col("municipality_province_name").alias("province_name_raw"),  # the province name as given (replaced below)
            F.col("municipality_category").alias("category"),  # A metro, B local, C district
            F.col("municipality_miif_category").alias("miif_category"),  # Treasury's finer size grouping
            F.col("municipality_parent_code").alias("parent_code")))  # for local municipalities, the district they belong to

NAMES = {"EC": "Eastern Cape", "FS": "Free State", "GT": "Gauteng", "KZN": "KwaZulu-Natal", "LP": "Limpopo",  # one official spelling per province code...
         "MP": "Mpumalanga", "NC": "Northern Cape", "NW": "North West", "WC": "Western Cape"}  # ...because the source spells KwaZulu-Natal two ways
names = F.create_map(*[F.lit(x) for kv in NAMES.items() for x in kv])  # turn that dictionary into a Spark lookup
clean_muni = clean_muni.withColumn("province_name", names[F.col("province_code")]).drop("province_name_raw")  # set the name from the code, and drop the old name column
clean_muni.write.mode("overwrite").format("delta").saveAsTable("clean_municipalities")  # save as a Delta table in the Lakehouse
display(clean_muni.groupBy("province_code", "province_name").count().orderBy("province_code"))  # show municipalities per province, to check there are 9

## 2. Audit opinions: add an ordinal severity rank

In [ ]:
RANK = {"unqualified": 1, "unqualified_emphasis_of_matter": 2, "qualified": 3, "adverse": 4, "disclaimer": 5}  # severity: 1 = clean audit ... 5 = disclaimer (worst); "outstanding" gets no rank
rank = F.create_map(*[F.lit(x) for kv in RANK.items() for x in kv])  # turn that dictionary into a Spark lookup

clean_audit = (raw["audit_opinions"]  # start from the raw audit opinions table
    .select(F.col("demarcation_code"),  # municipality code
            F.col("financial_year_end_year").alias("year"),  # financial year, with a shorter name
            F.col("opinion_code"),  # opinion type, e.g. qualified
            F.col("opinion_label"),  # opinion in words
            F.col("opinion_report_url").alias("report_url"))  # link to the audit report
    .withColumn("opinion_rank", rank[F.col("opinion_code")])  # add the 1-5 severity rank
    .withColumn("is_outstanding", F.col("opinion_code") == "outstanding")  # True if the audit wasn't finished
    .withColumn("is_clean", F.col("opinion_code") == "unqualified"))  # True for a clean audit
clean_audit.write.mode("overwrite").format("delta").saveAsTable("clean_audit_opinions")  # save as a Delta table

## 3. Irregular, fruitless and unauthorised expenditure: blanks → 0 with a flag, one row per municipality-year

In [ ]:
u = (raw["uifwexp"]  # start from the raw irregular, fruitless, wasteful and unauthorised expenditure table
     .select("demarcation_code",  # municipality code
             F.col("financial_year_end_year").alias("year"),  # financial year
             F.col("item_code").alias("item"),  # which type: irregular, fruitless or unauthorised
             "amount")  # the rand amount
     .withColumn("was_blank", F.col("amount").isNull())  # decision 7: remember which amounts were blank...
     .withColumn("amount", F.coalesce(F.col("amount"), F.lit(0.0))))  # ...then replace blanks with 0

clean_uifw = (u.groupBy("demarcation_code", "year")  # one row per municipality-year...
    .pivot("item", ["irregular", "fruitless", "unauthorised"])  # ...with separate columns for each expenditure type
    .agg(F.first("amount").alias("amt"), F.first("was_blank").alias("was_blank")))  # each type gets an amount column and a was_blank column
clean_uifw = clean_uifw.toDF(*[c.replace("_amt", "_exp") for c in clean_uifw.columns])  # rename e.g. irregular_amt to irregular_exp
clean_uifw = clean_uifw.withColumn("uifw_total", F.col("irregular_exp") + F.col("fruitless_exp") + F.col("unauthorised_exp"))  # add the three types into a total
clean_uifw.write.mode("overwrite").format("delta").saveAsTable("clean_uifw")  # save as a Delta table
clean_uifw.printSchema()  # show the columns and their types

## 4. Audited financial figures: keep every amount type, tidy names, flag availability

In [ ]:
def tidy(df, value_cols):
    """Give a financial table short, consistent column names. No rows are removed."""
    out = df.select(F.col("demarcation_code"),  # municipality code
                    F.col("financial_year_end_year").alias("year"),  # financial year
                    F.col("item_code"),  # line item code, e.g. 4400 = total expenditure
                    F.col("item_label"),  # line item in words
                    F.col("amount_type_code").alias("amount_type"),  # AUDA = audited actual, ORGB = original budget, and so on
                    *[F.col(c).alias(c.replace("_sum", "")) for c in value_cols])  # the number columns, dropping the "_sum" ending
    return out  # hand back the tidied table

clean_incexp = tidy(raw["incexp_v2"], ["amount_sum"])  # tidy the income and expenditure table
clean_fpos = tidy(raw["financial_position_v2"], ["amount_sum"])  # tidy the balance sheet table
clean_debt = tidy(raw["aged_debtor_v2"], [c for c in raw["aged_debtor_v2"].columns if c.endswith("_sum")])  # tidy the debtors table (it has several number columns)
for name, df in [("clean_incexp", clean_incexp), ("clean_financial_position", clean_fpos), ("clean_aged_debtor", clean_debt)]:  # go through the three tidied tables
    df.write.mode("overwrite").format("delta").saveAsTable(name)  # save each as a Delta table
    print(name, f"{df.count():,} rows")  # report its row count

def audited_flag(df, col):
    """List the municipality-years that have audited (AUDA) figures, marked True in a new column."""
    return (df.where(F.col("amount_type") == "AUDA")  # keep audited actual figures only
              .select("demarcation_code", "year")  # we only need who and when
              .distinct()  # one row per municipality-year
              .withColumn(col, F.lit(True)))  # mark them True

## 5. The panel: one row per municipality per year with an audit opinion

In [ ]:
panel = (clean_audit  # start from audit opinions: one row per municipality-year
    .join(clean_muni.select("demarcation_code", "name", "province_code", "category"), "demarcation_code", "left")  # attach name, province and type
    .join(clean_uifw, ["demarcation_code", "year"], "left")  # attach irregular, fruitless and unauthorised expenditure
    .join(audited_flag(clean_incexp, "has_audited_incexp"), ["demarcation_code", "year"], "left")  # attach: has audited income statement?
    .join(audited_flag(clean_fpos, "has_audited_fin_position"), ["demarcation_code", "year"], "left")  # attach: has audited balance sheet?
    .join(audited_flag(clean_debt, "has_audited_debtors"), ["demarcation_code", "year"], "left")  # attach: has audited debtors?
    .withColumn("mscoa_era", F.col("year") >= 2018)  # decision 8: True from 2018, when the new chart of accounts started
    # The flags only make sense in years where the source table exists (financials from 2018, debtors from 2020).
    # In those years a missing match becomes False (decision 5: missing is a risk flag); earlier years stay empty.
    .withColumn("has_audited_incexp", F.when(F.col("year") >= 2018, F.coalesce("has_audited_incexp", F.lit(False))))  # income statement flag, 2018+
    .withColumn("has_audited_fin_position", F.when(F.col("year") >= 2018, F.coalesce("has_audited_fin_position", F.lit(False))))  # balance sheet flag, 2018+
    .withColumn("has_audited_debtors", F.when(F.col("year") >= 2020, F.coalesce("has_audited_debtors", F.lit(False)))))  # debtors flag, 2020+
panel.write.mode("overwrite").format("delta").saveAsTable("clean_muni_year_panel")  # save the panel as a Delta table
print(f"clean_muni_year_panel: {panel.count():,} rows, {len(panel.columns)} columns")  # report its size

## 6. Reconciliation checks

In [ ]:
def check(name, ok, detail=""):
    """Print PASS or FAIL for one check, with optional detail in brackets."""
    print(("PASS " if ok else "FAIL ") + name + (f"  ({detail})" if detail else ""))  # e.g. "PASS no duplicate municipality-years"

p = spark.table("clean_muni_year_panel")  # open the panel we just saved
n_raw = raw["audit_opinions"].count()  # number of raw audit opinions
check("panel has one row per audit opinion", p.count() == n_raw, f"{p.count()} vs {n_raw}")  # check 1: no rows lost or added

dups = p.groupBy("demarcation_code", "year").count().where("count > 1").count()  # count municipality-years that appear more than once
check("no duplicate municipality-years", dups == 0)  # check 2: should be none

provs = [r[0] for r in spark.table("clean_municipalities").select("province_code").distinct().collect()]  # the distinct province codes after the fix
check("9 province codes", len(provs) == 9, sorted(provs))  # check 3: South Africa has 9 provinces

blanks = raw["uifwexp"].where(F.col("amount").isNull()).count()  # number of blank amounts in the raw data
blank_cols = ["irregular_was_blank", "fruitless_was_blank", "unauthorised_was_blank"]  # the three flag columns
flags_per_row = sum(F.col(c).cast("int") for c in blank_cols)  # per row, add up the flags (True counts as 1)
flagged = spark.table("clean_uifw").select(flags_per_row.alias("n")).agg(F.sum("n")).first()[0]  # total flags across the whole table
check("every blank expenditure amount is flagged", blanks == flagged, f"{blanks} blanks, {flagged} flagged")  # check 4: every blank is flagged

raw_total = raw["uifwexp"].agg(F.sum("amount")).first()[0]  # rand total in the raw table (blanks ignored)
clean_total = spark.table("clean_uifw").agg(F.sum("uifw_total")).first()[0]  # rand total after recoding blanks to 0
check("expenditure total unchanged by recoding", abs(raw_total - clean_total) < 1, f"R{raw_total:,.0f}")  # check 5: recoding mustn't change the total

miss = p.where("year BETWEEN 2018 AND 2024 AND NOT has_audited_incexp").count()  # municipality-years 2018-2024 without an audited income statement
check("112 municipality-years 2018-2024 without audited income statement", miss == 112, miss)  # check 6: matches the missingness report

same_rows = (spark.table("clean_incexp").count() == raw["incexp_v2"].count()  # income table: same rows before and after
             and spark.table("clean_financial_position").count() == raw["financial_position_v2"].count()  # balance sheet: same rows
             and spark.table("clean_aged_debtor").count() == raw["aged_debtor_v2"].count())  # debtors: same rows
check("no rows dropped from financial tables", same_rows)  # check 7: tidying removed nothing